# Final Competition-Grade GlyphMatics + RC9IMG Tinker Submission — 100% Restore Tailguard

**Purpose:** push beyond the 0.86 plateau by keeping **100% GlyphMatics Frobenius restore** while changing only the spectral transport distribution.

Confirmed empirical anchors:

- Flat 1.18/1.20 cap: about `0.86`
- 1.22 cap: about `0.85`
- Dual-pair split: about `0.85`
- Uniform 100% restore: about `0.86`

## New one-change test

Uniform 100% restore amplified all retained rank-32 singular directions equally. This variant keeps exact 100% energy restore, but damps the weakest retained tail directions before the final energy restore:

```text
rank 32 -> head 24 full weight + tail 8 damped to 0.88 -> global rescale to exact 100% Frobenius energy
```

This is still a 100% restore notebook. It just redirects restored energy away from weak/noisy tail singulars and into the dominant transport path.

Default submission contract:

```text
FORCED_FUSED_RANK = 32
GLYPHMATIC_RESTORE_RATIO = 1.00
GLYPHMATIC_HEAD_RANK = 24
GLYPHMATIC_TAIL_FACTOR = 0.88
GLYPHMATIC_SCALE_HARD_CAP = 0
transport = glyphmatic_rank32_frobenius_100_restore_tailguard
```

**Kaggle accelerator:** `GPU T4 x2`


**Combined addition:** after `submission.zip` is created, this notebook renders a verified RC9IMG PNG container that can reconstruct the submission zip from the image alone.


## Required Kaggle Inputs

Attach these inputs before running:

1. Competition input: `NVIDIA Nemotron Model Reasoning Challenge`.
2. Base model: `nemotron-3-nano-30b-a3b-bf16`.
3. Adapter input: `huikang/nemotron-adapter`.
4. Tinker wheelhouse input containing `tinker`, `tinker-cookbook`, and `chz` wheels.

Internet is **not required** when these inputs are attached.


In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import json
import importlib.util

print("Python:", sys.version)
print("Working dir:", Path.cwd())

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

if not KAGGLE_INPUT.exists():
    raise RuntimeError("This notebook is intended to run inside Kaggle.")

KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)

def print_inputs():
    print("\n[Inputs]")
    for p in sorted(KAGGLE_INPUT.iterdir()):
        print(" -", p)

print_inputs()


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Working dir: /kaggle/working

[Inputs]
 - /kaggle/input/competitions
 - /kaggle/input/datasets
 - /kaggle/input/models


## 1. Install or load Tinker locally

This scans every Kaggle input for local wheels and installs `tinker-cookbook` without using the Internet. If `tinker_cookbook` is already installed, it uses the existing installation.


In [2]:
from pathlib import Path
import os
import subprocess
import sys
import importlib.util

def list_wheel_dirs():
    rows = []
    for root in [Path("/kaggle/input"), Path("/kaggle/working"), Path("/tmp")]:
        if not root.exists():
            continue
        for d in [root] + [p for p in root.rglob("*") if p.is_dir()]:
            wheels = sorted(d.glob("*.whl"))
            if wheels:
                rows.append((d, [w.name for w in wheels]))
    return rows

def score_tinker_dir(names):
    low = " ".join(n.lower() for n in names)
    score = 0
    for token in ["tinker_cookbook", "tinker-cookbook", "tinker_", "tinker-", "chz"]:
        if token in low:
            score += 1
    return score

def find_tinker_wheelhouse():
    candidates = []
    for d, names in list_wheel_dirs():
        score = score_tinker_dir(names)
        if score:
            candidates.append((score, len(names), d, names))
    candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
    return candidates[0] if candidates else None

print("[Tinker] scanning wheel folders...")
for d, names in list_wheel_dirs():
    interesting = [n for n in names if ("tinker" in n.lower() or "chz" in n.lower())]
    if interesting:
        print("\n[wheel-dir]", d)
        for name in interesting:
            print(" -", name)

if importlib.util.find_spec("tinker_cookbook") is not None:
    print("[Tinker] tinker_cookbook already installed; skipping wheel install")
else:
    explicit = os.environ.get("WHEEL_DIR")
    candidate = None

    if explicit and Path(explicit).exists():
        candidate = (999, 0, Path(explicit), [p.name for p in Path(explicit).glob("*.whl")])
    else:
        candidate = find_tinker_wheelhouse()

    if candidate is None:
        raise FileNotFoundError(
            "Could not find local tinker wheelhouse. Attach a Kaggle input containing "
            "tinker-cookbook/tinker/chz wheels or set WHEEL_DIR to that folder."
        )

    _, _, wheel_dir, names = candidate
    print("[Tinker] selected wheel_dir:", wheel_dir)

    cmd = [
        sys.executable, "-m", "pip", "install",
        "--no-index",
        f"--find-links={wheel_dir}",
        "tinker-cookbook",
        "tinker",
    ]
    print("[Tinker] pip:", " ".join(cmd))
    subprocess.run(cmd, check=True)

import importlib.metadata as md
import tinker_cookbook
from tinker_cookbook import weights

print("[Tinker] ready:", tinker_cookbook.__file__)
print("[Tinker] tinker_cookbook version:", md.version("tinker-cookbook"))
print("[Tinker] tinker version:", md.version("tinker"))
print("[Tinker] has build_lora_adapter:", hasattr(weights, "build_lora_adapter"))

if not hasattr(weights, "build_lora_adapter"):
    raise RuntimeError("tinker_cookbook.weights.build_lora_adapter not found")


[Tinker] scanning wheel folders...

[wheel-dir] /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse
 - chz-0.4.0-py3-none-any.whl
 - tinker-0.18.1-py3-none-any.whl
 - tinker_cookbook-0.3.0-py3-none-any.whl
[Tinker] selected wheel_dir: /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse
[Tinker] pip: /usr/bin/python3 -m pip install --no-index --find-links=/kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse tinker-cookbook tinker
Looking in links: /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse
Processing /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse/tinker_cookbook-0.3.0-py3-none-any.whl
Processing /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse/tinker-0.18.1-py3-none-any.whl
Processing /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse/chz-0.4.0-py3-none-any.whl (from tinker-cookbook)
[Tinker] ready: /usr/local/lib/python3.12/dist-packages/tinker_cookbook/__init__.py
[Tinker] tinker_cookbook version: 0.3.0
[T

## 2. Detect local model and adapter paths

This avoids `snapshot_download(...)` and keeps the run fully offline.


In [3]:
from pathlib import Path
import json

def find_first_existing(paths):
    for p in paths:
        if Path(p).exists():
            return str(Path(p))
    return None

ADAPTER_PATH_CANDIDATES = [
    "/kaggle/input/models/huikang/nemotron-adapter/transformers/default/20",
    "/kaggle/input/huikang/nemotron-adapter/transformers/default/20",
    "/kaggle/input/nemotron-adapter/transformers/default/20",
    "/kaggle/input/nemotron-adapter",
]

BASE_MODEL_CANDIDATES = [
    "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    "/kaggle/input/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    "/kaggle/input/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    "/kaggle/input/nemotron-3-nano-30b-a3b-bf16",
]

ADAPTER_PATH = find_first_existing(ADAPTER_PATH_CANDIDATES)
BASE_MODEL_PATH = find_first_existing(BASE_MODEL_CANDIDATES)

if ADAPTER_PATH is None:
    # Generic fallback: adapter folder should contain adapter_config.json and adapter_model.safetensors.
    hits = []
    for cfg in Path("/kaggle/input").rglob("adapter_config.json"):
        folder = cfg.parent
        if (folder / "adapter_model.safetensors").exists():
            if "nemotron" in str(folder).lower() or "adapter" in str(folder).lower():
                hits.append(folder)
    if hits:
        ADAPTER_PATH = str(sorted(hits, key=lambda p: len(str(p)))[0])

if BASE_MODEL_PATH is None:
    # Generic fallback: local base model folder should contain config.json.
    hits = []
    for cfg in Path("/kaggle/input").rglob("config.json"):
        folder = cfg.parent
        s = str(folder).lower()
        if "nemotron" in s and ("30b" in s or "nano" in s):
            hits.append(folder)
    if hits:
        BASE_MODEL_PATH = str(sorted(hits, key=lambda p: len(str(p)))[0])

if ADAPTER_PATH is None:
    raise FileNotFoundError("Nemotron adapter input not found. Attach huikang/nemotron-adapter.")

if BASE_MODEL_PATH is None:
    raise FileNotFoundError("Local Nemotron base model not found. Attach nemotron-3-nano-30b-a3b-bf16.")

print("[Paths] ADAPTER_PATH:", ADAPTER_PATH)
print("[Paths] BASE_MODEL_PATH:", BASE_MODEL_PATH)

# Basic sanity checks.
if not (Path(ADAPTER_PATH) / "adapter_config.json").exists():
    raise FileNotFoundError(f"adapter_config.json missing under {ADAPTER_PATH}")

if not (Path(BASE_MODEL_PATH) / "config.json").exists():
    raise FileNotFoundError(f"config.json missing under {BASE_MODEL_PATH}")


[Paths] ADAPTER_PATH: /kaggle/input/models/huikang/nemotron-adapter/transformers/default/20
[Paths] BASE_MODEL_PATH: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1


## 3. Apply GlyphMatics 100% restore tailguard patch

This patch preserves evaluator-compatible rank 32 output while forcing the compressed fused projection to carry **100% of the original full-rank Frobenius energy**.

The difference from the previous 100% restore is spectral shaping:

```python
FORCED_FUSED_RANK = 32
GLYPHMATIC_RESTORE_RATIO = 1.00
GLYPHMATIC_HEAD_RANK = 24
GLYPHMATIC_TAIL_FACTOR = 0.88
GLYPHMATIC_SCALE_HARD_CAP = 0
```

Transport rule:

```text
top 24 singular directions: full weight
tail 8 singular directions: 0.88 weight
then globally rescale to exact 100% Frobenius energy
```

This tests whether the 0.86 plateau is caused by tail-direction noise rather than insufficient total energy.


In [4]:
from __future__ import annotations

from collections import Counter
import json
import os
from pathlib import Path

import torch
import tinker_cookbook.weights._adapter as A

FORCED_FUSED_RANK = int(os.environ.get("FORCED_FUSED_RANK", "32"))
GLYPHMATIC_RESTORE_RATIO = float(os.environ.get("GLYPHMATIC_RESTORE_RATIO", "1.00"))
GLYPHMATIC_HEAD_RANK = int(os.environ.get("GLYPHMATIC_HEAD_RANK", "24"))
GLYPHMATIC_TAIL_FACTOR = float(os.environ.get("GLYPHMATIC_TAIL_FACTOR", "0.88"))
# 0 means no hard cap on the post-shape global scale. Leave disabled for exact 100% restore.
GLYPHMATIC_SCALE_HARD_CAP = float(os.environ.get("GLYPHMATIC_SCALE_HARD_CAP", "0"))

if FORCED_FUSED_RANK <= 0:
    raise ValueError("FORCED_FUSED_RANK must be positive")
if GLYPHMATIC_RESTORE_RATIO <= 0:
    raise ValueError("GLYPHMATIC_RESTORE_RATIO must be positive")
if not (0 < GLYPHMATIC_TAIL_FACTOR <= 1.0):
    raise ValueError("GLYPHMATIC_TAIL_FACTOR must be in (0, 1]")
if GLYPHMATIC_SCALE_HARD_CAP < 0:
    raise ValueError("GLYPHMATIC_SCALE_HARD_CAP must be >= 0")


class GlyphmaticTransportLedger:
    def __init__(self):
        self.events = []
        self.counts = Counter()

    def emit(self, *, src, dst, op, alpha, beta=None, gamma=None):
        event = {
            "alpha": alpha,
            "source": str(src),
            "dest": str(dst),
            "op": str(op),
            "beta": beta or {},
            "gamma": gamma or {},
        }
        self.events.append(event)
        self.counts[(alpha, op)] += 1

    def markdown(self) -> str:
        lines = [
            "# GlyphMatics Transport Ledger",
            "",
            "Generated during tinker-cookbook adapter conversion.",
            "",
            "## Submission configuration",
            "",
            f"- `FORCED_FUSED_RANK`: `{FORCED_FUSED_RANK}`",
            f"- `GLYPHMATIC_RESTORE_RATIO`: `{GLYPHMATIC_RESTORE_RATIO}`",
            f"- `GLYPHMATIC_HEAD_RANK`: `{GLYPHMATIC_HEAD_RANK}`",
            f"- `GLYPHMATIC_TAIL_FACTOR`: `{GLYPHMATIC_TAIL_FACTOR}`",
            f"- `GLYPHMATIC_SCALE_HARD_CAP`: `{GLYPHMATIC_SCALE_HARD_CAP}`",
            "- `transport`: `glyphmatic_rank32_frobenius_100_restore_tailguard`",
            "",
            "## Coordinate definition",
            "",
            "- **α**: module/transport family.",
            "- **β**: tensor geometry.",
            "- **γ**: preservation/compression action.",
            "",
            "## Event summary",
            "",
            "| α | op | count |",
            "|---|---:|---:|",
        ]
        for (alpha, op), count in sorted(self.counts.items()):
            lines.append(f"| `{alpha}` | `{op}` | {count} |")

        lines += [
            "",
            "## First 60 events",
            "",
            "| # | α | op | source | destination | γ |",
            "|---:|---|---|---|---|---|",
        ]
        for i, event in enumerate(self.events[:60], 1):
            gamma = json.dumps(event["gamma"], sort_keys=True)
            lines.append(
                f"| {i} | `{event['alpha']}` | `{event['op']}` | "
                f"`{event['source']}` | `{event['dest']}` | `{gamma}` |"
            )
        return "\n".join(lines) + "\n"

    def print_summary(self):
        print("[GlyphMatics ledger] events:", len(self.events))
        for (alpha, op), count in sorted(self.counts.items()):
            print(f"[GlyphMatics ledger] {alpha}:{op}={count}")


GLYPH_LEDGER = GlyphmaticTransportLedger()


def _compress_lora_pair_to_rank(B: torch.Tensor, A_mat: torch.Tensor, rank: int):
    """
    Compress Delta = B @ A to rank-k, then perform GlyphMatics 100% restore
    with tail-guard spectral shaping.

    Difference from the previous uniform 100% restore:
      - top singular directions keep full shaping weight
      - tail directions are damped by GLYPHMATIC_TAIL_FACTOR
      - a global rescale restores exact Frobenius energy back to 100%

    This preserves rank-32 compatibility while redirecting energy away from
    weaker/noisier tail singular directions.
    """
    delta = B.float() @ A_mat.float()

    U, S, Vh = torch.linalg.svd(delta, full_matrices=False)
    total_mass = S.sum().clamp_min(1e-12)
    full_energy = torch.sqrt(torch.sum(S ** 2)).clamp_min(1e-12)

    U = U[:, :rank]
    S_k = S[:rank]
    Vh = Vh[:rank, :]

    kept_energy = torch.sqrt(torch.sum(S_k ** 2)).clamp_min(1e-12)

    head_rank = max(0, min(int(GLYPHMATIC_HEAD_RANK), int(rank)))
    shape = torch.ones_like(S_k)
    if head_rank < rank:
        shape[head_rank:] = GLYPHMATIC_TAIL_FACTOR

    shaped_energy = torch.sqrt(torch.sum((S_k * shape) ** 2)).clamp_min(1e-12)
    raw_global_scale = (full_energy * GLYPHMATIC_RESTORE_RATIO) / shaped_energy

    if GLYPHMATIC_SCALE_HARD_CAP > 0:
        global_scale = torch.clamp(raw_global_scale, min=0.0, max=GLYPHMATIC_SCALE_HARD_CAP)
        scale_policy = f"hard_cap_{GLYPHMATIC_SCALE_HARD_CAP}"
    else:
        global_scale = raw_global_scale
        scale_policy = "uncapped_exact_restore_after_tailguard"

    S_eff = S_k * shape * global_scale
    S_eff = torch.clamp(S_eff, min=0.0)

    # Factor U diag(S_eff) Vh as LoRA-compatible B_new @ A_new.
    sroot = torch.sqrt(S_eff.clamp_min(1e-30))
    B_new = U * sroot.unsqueeze(0)
    A_new = sroot.unsqueeze(1) * Vh

    restored_energy = torch.sqrt(torch.sum(S_eff ** 2)).clamp_min(1e-12)
    stats = {
        "rank_in": int(B.shape[1]),
        "rank_out": int(rank),
        "singular_mass_kept": float(S_k.sum() / total_mass),
        "frobenius_energy_kept_ratio": float(kept_energy / full_energy),
        "target_restore_ratio": float(GLYPHMATIC_RESTORE_RATIO),
        "head_rank": int(head_rank),
        "tail_rank": int(rank - head_rank),
        "tail_factor": float(GLYPHMATIC_TAIL_FACTOR),
        "pre_shape_energy_ratio": float(shaped_energy / full_energy),
        "raw_global_scale": float(raw_global_scale),
        "applied_global_scale": float(global_scale),
        "restored_energy_ratio": float(restored_energy / full_energy),
        "scale_policy": scale_policy,
        "preservation": "rank32_glyphmatic_frobenius_100_restore_tailguard",
        "lost_subspace_note": "rank32 restores energy magnitude only, not discarded directions",
    }

    return B_new.to(B.dtype).contiguous(), A_new.to(A_mat.dtype).contiguous(), stats


def patched_merge_fused_projections(
    fused_model_key: str,
    adapter_layer_prefix: str,
    components,
    model_state_shapes,
    peft_weights,
    target_modules,
    profile,
) -> int:
    fused_out_dim = model_state_shapes[fused_model_key][0]
    fused_target_name = fused_model_key.removesuffix(".weight").rsplit(".", 1)[-1]

    component_order = None
    for target, comps in profile.fused_projection_map:
        if target == fused_target_name:
            component_order = comps
            break
    assert component_order is not None

    comp_by_name = {name: (lora_A, lora_B) for name, lora_A, lora_B in components}

    lora_A_parts = []
    comp_slices = []
    merged_rank = 0
    row_offset = 0

    for comp_name in component_order:
        if comp_name not in comp_by_name:
            raise RuntimeError(
                f"Missing component {comp_name!r} for fused target {fused_model_key!r}"
            )

        lora_A, lora_B = comp_by_name[comp_name]
        r = lora_A.shape[0]
        out_dim = lora_B.shape[0]

        lora_A_parts.append(lora_A)
        comp_slices.append((row_offset, row_offset + out_dim, r, comp_name))
        row_offset += out_dim
        merged_rank += r

    merged_lora_A = torch.cat(lora_A_parts, dim=0)
    merged_lora_B = torch.zeros(
        fused_out_dim,
        merged_rank,
        dtype=merged_lora_A.dtype,
        device=merged_lora_A.device,
    )

    rank_offset = 0
    for row_start, row_end, r, comp_name in comp_slices:
        _, lora_B = comp_by_name[comp_name]
        merged_lora_B[row_start:row_end, rank_offset:rank_offset + r] = lora_B
        rank_offset += r

    final_rank = merged_rank
    compression_stats = {
        "rank_in": int(merged_rank),
        "rank_out": int(merged_rank),
        "preservation": "exact_no_rank_compression_needed",
    }

    if merged_rank > FORCED_FUSED_RANK:
        merged_lora_B, merged_lora_A, svd_stats = _compress_lora_pair_to_rank(
            merged_lora_B,
            merged_lora_A,
            FORCED_FUSED_RANK,
        )
        final_rank = FORCED_FUSED_RANK
        compression_stats = svd_stats

    peft_target_key = f"{adapter_layer_prefix}.{fused_target_name}.weight"

    GLYPH_LEDGER.emit(
        src=f"{adapter_layer_prefix}.{{{','.join(component_order)}}}",
        dst=peft_target_key,
        op="glyphmatic_fused_projection_transport",
        alpha="glyphmatic_frobenius_restore_tailguard",
        beta={
            "fused_out_dim": int(fused_out_dim),
            "component_count": len(component_order),
            "component_order": list(component_order),
        },
        gamma=compression_stats,
    )

    A._add_peft_weight(peft_target_key, merged_lora_A, merged_lora_B, peft_weights, target_modules)
    return final_rank


A._merge_fused_projections = patched_merge_fused_projections

print("[GlyphMatics] patched:", A._merge_fused_projections.__name__)
print("[GlyphMatics] FORCED_FUSED_RANK:", FORCED_FUSED_RANK)
print("[GlyphMatics] GLYPHMATIC_RESTORE_RATIO:", GLYPHMATIC_RESTORE_RATIO)
print("[GlyphMatics] GLYPHMATIC_HEAD_RANK:", GLYPHMATIC_HEAD_RANK)
print("[GlyphMatics] GLYPHMATIC_TAIL_FACTOR:", GLYPHMATIC_TAIL_FACTOR)
print("[GlyphMatics] GLYPHMATIC_SCALE_HARD_CAP:", GLYPHMATIC_SCALE_HARD_CAP)
print("[GlyphMatics] transport: glyphmatic_rank32_frobenius_100_restore_tailguard")


[GlyphMatics] patched: patched_merge_fused_projections
[GlyphMatics] FORCED_FUSED_RANK: 32
[GlyphMatics] SVD_ENERGY_GAIN_CAP: 1.205


## 4. Build adapter package

This clears stale output, builds the adapter using the local base model, writes the transport ledger, writes a compact manifest, and guarantees the completion marker exists.


In [5]:
from pathlib import Path
import shutil
import json
from tinker_cookbook import weights

OUTPUT_DIR = Path("/kaggle/working/nemotron-adapter-ready-to-submit")

# Deterministic rerun: tinker refuses to write into an existing output folder.
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

print("[Build] adapter_path:", ADAPTER_PATH)
print("[Build] base_model_path:", BASE_MODEL_PATH)
print("[Build] output_dir:", OUTPUT_DIR)

weights.build_lora_adapter(
    base_model=str(BASE_MODEL_PATH),
    adapter_path=str(ADAPTER_PATH),
    output_path=str(OUTPUT_DIR),
)

# Required marker. Some tinker-cookbook versions do not emit it.
if (OUTPUT_DIR / "adapter_config.json").exists() and (OUTPUT_DIR / "adapter_model.safetensors").exists():
    (OUTPUT_DIR / "checkpoint_complete").write_text("ok\n", encoding="utf-8")
else:
    raise FileNotFoundError("Adapter build did not produce adapter_config.json and adapter_model.safetensors")

# Review/debug metadata; harmless for human inspection.
GLYPH_LEDGER.print_summary()
ledger_text = GLYPH_LEDGER.markdown()
(OUTPUT_DIR / "GLYPHMATICS_TRANSPORT_LEDGER.md").write_text(ledger_text, encoding="utf-8")

manifest = {
    "submission_variant": "glyphmatics_rank32_frobenius_100_restore_tailguard",
    "forced_fused_rank": int(FORCED_FUSED_RANK),
    "glyphmatic_restore_ratio": float(GLYPHMATIC_RESTORE_RATIO),
    "glyphmatic_head_rank": int(GLYPHMATIC_HEAD_RANK),
    "glyphmatic_tail_factor": float(GLYPHMATIC_TAIL_FACTOR),
    "glyphmatic_scale_hard_cap": float(GLYPHMATIC_SCALE_HARD_CAP),
    "transport": "glyphmatic_rank32_frobenius_100_restore_tailguard",
    "restore_definition": "damp retained tail singular directions, then globally rescale so compressed delta Frobenius energy equals 100% of full fused delta energy",
    "rank_limit_note": "100% energy restore does not restore discarded rank directions",
    "adapter_path": str(ADAPTER_PATH),
    "base_model_path": str(BASE_MODEL_PATH),
}
(OUTPUT_DIR / "GLYPHMATICS_SUBMISSION_MANIFEST.json").write_text(
    json.dumps(manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

readme = OUTPUT_DIR / "README.md"
if readme.exists():
    with open(readme, "a", encoding="utf-8") as f:
        f.write("\n\n")
        f.write(ledger_text)
else:
    readme.write_text(
        "# Nemotron Adapter Submission\n\n" + ledger_text,
        encoding="utf-8",
    )

print("[Build] output files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name, p.stat().st_size)


[Build] adapter_path: /kaggle/input/models/huikang/nemotron-adapter/transformers/default/20
[Build] base_model_path: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
[Build] output_dir: /kaggle/working/nemotron-adapter-ready-to-submit


MoE expert LoRA serving for nemotron models is experimental in vLLM and not yet supported in SGLang. The adapter will be produced but may not work with all serving configurations.


[GlyphMatics ledger] events: 23
[GlyphMatics ledger] mamba_or_fused_projection:fused_projection_transport=23
[Build] output files:
 - GLYPHMATICS_TRANSPORT_LEDGER.md 8177
 - README.md 8208
 - adapter_config.json 618
 - adapter_model.safetensors 3554384888
 - checkpoint_complete 3


## 5. Validate and create submission.zip

Only submit the resulting `/kaggle/working/submission.zip`.


In [6]:
import zipfile
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/nemotron-adapter-ready-to-submit")
ZIP_PATH = Path("/kaggle/working/submission.zip")

required = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "README.md",
    "checkpoint_complete",
]

missing = [name for name in required if not (OUTPUT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing required submission files: {missing}")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

# Keep submission minimal. Ledger and manifest are documentation only and should not affect adapter loading.
include = required + [
    "GLYPHMATICS_TRANSPORT_LEDGER.md",
    "GLYPHMATICS_SUBMISSION_MANIFEST.json",
]

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in include:
        p = OUTPUT_DIR / name
        if p.exists():
            zf.write(p, arcname=name)

print("[Zip] wrote:", ZIP_PATH)
print("[Zip] size:", ZIP_PATH.stat().st_size)
print("[Zip] contents:")
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    for name in zf.namelist():
        print(" -", name)

assert ZIP_PATH.exists() and ZIP_PATH.stat().st_size > 0


[Zip] wrote: /kaggle/working/submission.zip
[Zip] size: 3270303048
[Zip] contents:
 - adapter_config.json
 - adapter_model.safetensors
 - README.md
 - checkpoint_complete
 - GLYPHMATICS_TRANSPORT_LEDGER.md


## 6. RC9IMG image-only storage / transport container

This cell combines the competition adapter output with the RC9/RGS image-container path.

It creates and verifies:

```text
/kaggle/working/submission.zip
→ RGS-MVGC reversible compression
→ RC9 physical cube glyph transport
→ rendered PNG image container
→ decode PNG only
→ byte-perfect restored submission.zip
```

The Kaggle submission remains `submission.zip`. The RC9 image-only artifact is `submission.rc9img.png` for storage/transport/proof.


In [ ]:

from __future__ import annotations

import bz2
import hashlib
import json
import lzma
import math
import random
import struct
import time
import zlib
from pathlib import Path
from typing import Any, Dict, Iterable, List, Tuple

# ============================================================
# RC9IMG + RGS-MVGC combined image-only container
# Self-contained. No PIL. No internet. No sidecar required.
# ============================================================

RC9IMG_SYSTEM = "RC9IMG-RGS-MVGC-NEMOTRON"
RC9IMG_VERSION_MAJOR = 1
RC9IMG_MAGIC = b"R9IMGN01"
RC9IMG_CELL_W = 8
RC9IMG_CELL_H = 12
RC9IMG_CUBE_SIZE = 54
RC9IMG_PNG_SIG = b"\x89PNG\r\n\x1a\n"

# magic, version, key_len, source_len, rgs_len, bit_pad,
# cube_pad, cube_count, glyph_count, payload_glyph_count,
# source_sha, rgs_sha, payload_sha
RC9IMG_HDR = struct.Struct(">8sHHQQBHIQQ32s32s32s")
RC9IMG_HDR_GLYPHS = math.ceil(RC9IMG_HDR.size * 8 / 6)
RC9IMG_HDR_PAD = (-RC9IMG_HDR.size * 8) % 6

RC9IMG_FACE_ORDER = ["U", "D", "F", "B", "L", "R"]
RC9IMG_MOVE_TOKENS = [
    "U", "U'", "D", "D'", "F", "F'", "B", "B'",
    "L", "L'", "R", "R'", "U2", "D2", "F2", "B2", "L2", "R2",
]

RC9IMG_KING_WEN_BITS_BOTTOM_TO_TOP = [
    "111111", "000000", "100010", "010001", "111010", "010111", "010000", "000010",
    "111011", "110111", "111000", "000111", "101111", "111101", "001000", "000100",
    "100110", "011001", "110000", "000011", "100101", "101001", "000001", "100000",
    "100111", "111001", "100001", "011110", "010010", "101101", "001110", "011100",
    "001111", "111100", "000101", "101000", "101011", "110101", "001010", "010100",
    "110001", "100011", "111110", "011111", "000110", "011000", "010110", "011010",
    "101110", "011101", "100100", "001001", "001011", "110100", "101100", "001101",
    "011011", "110110", "010011", "110010", "110011", "001100", "101010", "010101",
]
RC9IMG_REV_KING = {v: i for i, v in enumerate(RC9IMG_KING_WEN_BITS_BOTTOM_TO_TOP)}

RGS_MAGIC = b"RGSMVGC1"
RGS_HEADER_STRUCT = struct.Struct(">8sI")


def rc9_sha256(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def rgs_pack_container(header: Dict[str, Any], payload: bytes) -> bytes:
    header = dict(header)
    header["payload_len"] = len(payload)
    h = json.dumps(header, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return RGS_HEADER_STRUCT.pack(RGS_MAGIC, len(h)) + h + payload


def rgs_unpack_container(blob: bytes) -> Tuple[Dict[str, Any], bytes]:
    if len(blob) < RGS_HEADER_STRUCT.size:
        raise ValueError("RGS-MVGC container too short")
    magic, header_len = RGS_HEADER_STRUCT.unpack(blob[:RGS_HEADER_STRUCT.size])
    if magic != RGS_MAGIC:
        raise ValueError("bad RGS-MVGC magic")
    h0 = RGS_HEADER_STRUCT.size
    h1 = h0 + header_len
    if len(blob) < h1:
        raise ValueError("truncated RGS-MVGC header")
    header = json.loads(blob[h0:h1].decode("utf-8"))
    payload = blob[h1:]
    if len(payload) != int(header["payload_len"]):
        raise ValueError("RGS-MVGC payload length mismatch")
    return header, payload


def rgs_candidate_total_len(raw: bytes, method: str, payload: bytes, meta: Dict[str, Any]) -> int:
    h = {
        "system": "RGS-MVGC",
        "version": "1.0",
        "method": method,
        "original_len": len(raw),
        "sha256": rc9_sha256(raw),
        "meta": meta,
        "created_unix": 0,
    }
    return len(rgs_pack_container(h, payload))


def rgs_compress(raw: bytes) -> Tuple[bytes, Dict[str, Any]]:
    """Reversible RGS-MVGC precompression. Picks the smallest standard-library method."""
    candidates: List[Tuple[str, bytes, Dict[str, Any]]] = [
        ("raw", raw, {}),
        ("zlib9", zlib.compress(raw, 9), {"level": 9}),
        ("bz2_9", bz2.compress(raw, compresslevel=9), {"level": 9}),
    ]

    try:
        candidates.append(("lzma_6", lzma.compress(raw, preset=6), {"preset": 6}))
    except Exception as exc:
        print("[RGS-MVGC] lzma skipped:", repr(exc))

    method, payload, meta = min(
        candidates,
        key=lambda c: rgs_candidate_total_len(raw, c[0], c[1], c[2]),
    )

    header = {
        "system": "RGS-MVGC",
        "version": "1.0",
        "method": method,
        "original_len": len(raw),
        "sha256": rc9_sha256(raw),
        "meta": meta,
        "created_unix": int(time.time()),
    }
    return rgs_pack_container(header, payload), header


def rgs_decompress(blob: bytes) -> Tuple[bytes, Dict[str, Any]]:
    header, payload = rgs_unpack_container(blob)
    method = header["method"]

    if method == "raw":
        raw = payload
    elif method.startswith("zlib"):
        raw = zlib.decompress(payload)
    elif method == "bz2_9":
        raw = bz2.decompress(payload)
    elif method == "lzma_6":
        raw = lzma.decompress(payload)
    else:
        raise ValueError(f"unknown RGS-MVGC method: {method}")

    if len(raw) != int(header["original_len"]):
        raise ValueError("RGS-MVGC original length mismatch")
    if rc9_sha256(raw) != header["sha256"]:
        raise ValueError("RGS-MVGC SHA-256 mismatch")
    return raw, header


def rc9_bytes_to_glyphs(data: bytes) -> Tuple[List[int], int]:
    bits = "".join(f"{b:08b}" for b in data)
    pad = (-len(bits)) % 6
    if pad:
        bits += "0" * pad
    return [int(bits[i:i + 6], 2) for i in range(0, len(bits), 6)], pad


def rc9_glyphs_to_bytes(glyphs: List[int], pad: int = 0) -> bytes:
    bits = "".join(f"{int(g) & 0x3F:06b}" for g in glyphs)
    if pad:
        bits = bits[:-pad]
    if len(bits) % 8:
        bits += "0" * ((-len(bits)) % 8)
    return bytes(int(bits[i:i + 8], 2) for i in range(0, len(bits), 8))


def rc9_pack6(vals: List[int]) -> bytes:
    out = bytearray()
    acc = 0
    nbits = 0
    for v in vals:
        iv = int(v)
        if not 0 <= iv <= 63:
            raise ValueError(f"glyph out of range: {iv}")
        acc = (acc << 6) | iv
        nbits += 6
        while nbits >= 8:
            nbits -= 8
            out.append((acc >> nbits) & 0xFF)
            acc &= (1 << nbits) - 1 if nbits else 0
    if nbits:
        out.append((acc << (8 - nbits)) & 0xFF)
    return bytes(out)


def rc9_flat_to_state(vals: List[int]) -> Dict[str, List[List[int]]]:
    if len(vals) != RC9IMG_CUBE_SIZE:
        raise ValueError(f"cube requires {RC9IMG_CUBE_SIZE} glyphs; got {len(vals)}")
    out: Dict[str, List[List[int]]] = {}
    k = 0
    for face in RC9IMG_FACE_ORDER:
        out[face] = []
        for _ in range(3):
            out[face].append([int(vals[k]), int(vals[k + 1]), int(vals[k + 2])])
            k += 3
    return out


def rc9_state_to_flat(state: Dict[str, List[List[int]]]) -> List[int]:
    return [int(cell) for f in RC9IMG_FACE_ORDER for row in state[f] for cell in row]


def rc9_glyphs_to_cubes(glyphs: List[int]) -> Tuple[List[Dict[str, List[List[int]]]], int]:
    vals = [int(g) & 0x3F for g in glyphs]
    pad = (-len(vals)) % RC9IMG_CUBE_SIZE
    if pad:
        vals += [0] * pad
    return [
        rc9_flat_to_state(vals[i:i + RC9IMG_CUBE_SIZE])
        for i in range(0, len(vals), RC9IMG_CUBE_SIZE)
    ], pad


def rc9_cubes_to_glyphs(cubes: List[Dict[str, List[List[int]]]], cube_pad_glyphs: int = 0) -> List[int]:
    vals: List[int] = []
    for cube in cubes:
        vals.extend(rc9_state_to_flat(cube))
    if cube_pad_glyphs:
        vals = vals[:-cube_pad_glyphs]
    return vals


def rc9_face_row_col_to_sticker(face: str, row: int, col: int) -> Tuple[Tuple[int, int, int], Tuple[int, int, int]]:
    if face == "U":
        return ((col - 1, 1, row - 1), (0, 1, 0))
    if face == "D":
        return ((col - 1, -1, 1 - row), (0, -1, 0))
    if face == "F":
        return ((col - 1, 1 - row, 1), (0, 0, 1))
    if face == "B":
        return ((1 - col, 1 - row, -1), (0, 0, -1))
    if face == "L":
        return ((-1, 1 - row, col - 1), (-1, 0, 0))
    if face == "R":
        return ((1, 1 - row, 1 - col), (1, 0, 0))
    raise ValueError(face)


def rc9_sticker_to_face_row_col(pos: Tuple[int, int, int], normal: Tuple[int, int, int]) -> Tuple[str, int, int]:
    x, y, z = pos
    if normal == (0, 1, 0):
        return "U", z + 1, x + 1
    if normal == (0, -1, 0):
        return "D", 1 - z, x + 1
    if normal == (0, 0, 1):
        return "F", 1 - y, x + 1
    if normal == (0, 0, -1):
        return "B", 1 - y, 1 - x
    if normal == (-1, 0, 0):
        return "L", 1 - y, z + 1
    if normal == (1, 0, 0):
        return "R", 1 - y, 1 - z
    raise ValueError(f"bad normal: {normal}")


def rc9_rot90_vec(v: Tuple[int, int, int], axis: str, turns: int) -> Tuple[int, int, int]:
    turns %= 4
    x, y, z = v
    for _ in range(turns):
        if axis == "x":
            y, z = -z, y
        elif axis == "y":
            x, z = z, -x
        elif axis == "z":
            x, y = -y, x
        else:
            raise ValueError(axis)
    return x, y, z


RC9_MOVE_DEF = {
    "U": ("y", 1, 1),
    "D": ("y", -1, -1),
    "F": ("z", 1, 1),
    "B": ("z", -1, -1),
    "L": ("x", -1, -1),
    "R": ("x", 1, 1),
}


def rc9_parse_move(move: str) -> Tuple[str, int]:
    move = move.strip()
    if not move or move[0] not in RC9_MOVE_DEF:
        raise ValueError(f"bad move: {move!r}")
    base, suffix = move[0], move[1:]
    if suffix == "":
        return base, 1
    if suffix == "'":
        return base, -1
    if suffix == "2":
        return base, 2
    raise ValueError(f"bad move suffix: {move!r}")


def rc9_invert_move(move: str) -> str:
    base, amount = rc9_parse_move(move)
    if amount == 2:
        return base + "2"
    if amount == 1:
        return base + "'"
    return base


def rc9_invert_sequence(moves: Iterable[str]) -> List[str]:
    return [rc9_invert_move(m) for m in reversed(list(moves))]


def rc9_apply_move(state: Dict[str, List[List[int]]], move: str) -> Dict[str, List[List[int]]]:
    base, amount = rc9_parse_move(move)
    axis, layer, sign = RC9_MOVE_DEF[base]
    turns = sign * amount

    src = state
    dst = {f: [row[:] for row in src[f]] for f in RC9IMG_FACE_ORDER}

    for face in RC9IMG_FACE_ORDER:
        for row in range(3):
            for col in range(3):
                pos, normal = rc9_face_row_col_to_sticker(face, row, col)
                layer_coord = {"x": pos[0], "y": pos[1], "z": pos[2]}[axis]
                if layer_coord != layer:
                    continue
                npos = rc9_rot90_vec(pos, axis, turns)
                nnorm = rc9_rot90_vec(normal, axis, turns)
                nface, nrow, ncol = rc9_sticker_to_face_row_col(npos, nnorm)
                dst[nface][nrow][ncol] = src[face][row][col]

    return dst


def rc9_apply_sequence(state: Dict[str, List[List[int]]], moves: Iterable[str]) -> Dict[str, List[List[int]]]:
    out = state
    for m in moves:
        out = rc9_apply_move(out, m)
    return out


def rc9_generate_key(data_hash: str, cube_index: int = 0, length: int = 25) -> List[str]:
    seed_hex = hashlib.sha256(
        f"{RC9IMG_SYSTEM}|key|{data_hash}|cube={cube_index}|len={length}".encode()
    ).hexdigest()
    rng = random.Random(int(seed_hex, 16))
    out: List[str] = []
    prev_axis = None

    for _ in range(length):
        token = rng.choice(RC9IMG_MOVE_TOKENS)
        axis = token[0]
        while axis == prev_axis:
            token = rng.choice(RC9IMG_MOVE_TOKENS)
            axis = token[0]
        out.append(token)
        prev_axis = axis

    return out


def rc9_png_chunk(kind: bytes, data: bytes) -> bytes:
    return (
        struct.pack(">I", len(data))
        + kind
        + data
        + struct.pack(">I", zlib.crc32(kind + data) & 0xFFFFFFFF)
    )


def rc9_write_png_rgb(path: Path, width: int, height: int, rgb: bytes) -> None:
    if len(rgb) != width * height * 3:
        raise ValueError("RGB buffer size mismatch")
    raw = bytearray()
    stride = width * 3
    for y in range(height):
        raw.append(0)
        raw.extend(rgb[y * stride:(y + 1) * stride])
    ihdr = struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0)
    path.write_bytes(
        RC9IMG_PNG_SIG
        + rc9_png_chunk(b"IHDR", ihdr)
        + rc9_png_chunk(b"IDAT", zlib.compress(bytes(raw), 9))
        + rc9_png_chunk(b"IEND", b"")
    )


def rc9_read_png_rgb(path: Path) -> Tuple[int, int, bytes]:
    blob = path.read_bytes()
    if not blob.startswith(RC9IMG_PNG_SIG):
        raise ValueError("not a PNG file")

    pos = len(RC9IMG_PNG_SIG)
    width = height = None
    idat = bytearray()

    while pos < len(blob):
        if pos + 8 > len(blob):
            raise ValueError("truncated PNG chunk")
        ln = struct.unpack(">I", blob[pos:pos + 4])[0]
        kind = blob[pos + 4:pos + 8]
        data = blob[pos + 8:pos + 8 + ln]
        crc_given = struct.unpack(">I", blob[pos + 8 + ln:pos + 12 + ln])[0]
        crc_calc = zlib.crc32(kind + data) & 0xFFFFFFFF
        if crc_given != crc_calc:
            raise ValueError(f"PNG CRC mismatch in {kind!r}")
        pos += 12 + ln

        if kind == b"IHDR":
            width, height, bit_depth, color_type, comp, filt, interlace = struct.unpack(">IIBBBBB", data)
            if bit_depth != 8 or color_type != 2 or comp != 0 or filt != 0 or interlace != 0:
                raise ValueError("unsupported PNG format; use generated RC9IMG PNG")
        elif kind == b"IDAT":
            idat.extend(data)
        elif kind == b"IEND":
            break

    if width is None or height is None:
        raise ValueError("PNG missing IHDR")

    raw = zlib.decompress(bytes(idat))
    stride = width * 3
    out = bytearray()
    p = 0

    for _ in range(height):
        filter_type = raw[p]
        p += 1
        if filter_type != 0:
            raise ValueError("unsupported PNG filter; use generated RC9IMG PNG")
        out.extend(raw[p:p + stride])
        p += stride

    return width, height, bytes(out)


def rc9_draw_glyph_cell(img: bytearray, width: int, cell_x: int, cell_y: int, glyph: int) -> None:
    x0 = cell_x * RC9IMG_CELL_W
    y0 = cell_y * RC9IMG_CELL_H
    pattern_top = RC9IMG_KING_WEN_BITS_BOTTOM_TO_TOP[glyph][::-1]

    def set_px(x: int, y: int, color: Tuple[int, int, int]) -> None:
        idx = (y * width + x) * 3
        img[idx:idx + 3] = bytes(color)

    for yy in range(y0, y0 + RC9IMG_CELL_H):
        for xx in range(x0, x0 + RC9IMG_CELL_W):
            set_px(xx, yy, (255, 255, 255))

    for line_idx, bit in enumerate(pattern_top):
        yy0 = y0 + line_idx * 2
        ranges = [(0, RC9IMG_CELL_W)] if bit == "1" else [(0, 3), (5, RC9IMG_CELL_W)]
        for yy in (yy0, yy0 + 1):
            for a, b in ranges:
                for xx in range(x0 + a, x0 + b):
                    set_px(xx, yy, (0, 0, 0))


def rc9_decode_glyph_cell(rgb: bytes, width: int, cell_x: int, cell_y: int) -> int:
    x0 = cell_x * RC9IMG_CELL_W
    y0 = cell_y * RC9IMG_CELL_H
    bits_top = []
    for line_idx in range(6):
        xx = x0 + 4
        yy = y0 + line_idx * 2
        idx = (yy * width + xx) * 3
        r, g, b = rgb[idx], rgb[idx + 1], rgb[idx + 2]
        bits_top.append("1" if (r + g + b) < 384 else "0")
    bits_bottom = "".join(bits_top)[::-1]
    if bits_bottom not in RC9IMG_REV_KING:
        raise ValueError(f"unrecognized glyph pattern: {bits_bottom}")
    return RC9IMG_REV_KING[bits_bottom]


def rc9_render_glyph_png(path: Path, glyphs: List[int], cols: int = 192) -> Dict[str, Any]:
    if cols < 8:
        raise ValueError("cols must be >= 8")

    rows = math.ceil(len(glyphs) / cols)
    total_cells = rows * cols
    padded = glyphs + [0] * (total_cells - len(glyphs))

    width = cols * RC9IMG_CELL_W
    height = rows * RC9IMG_CELL_H
    img = bytearray([255] * (width * height * 3))

    for i, g in enumerate(padded):
        rc9_draw_glyph_cell(img, width, i % cols, i // cols, int(g))

    rc9_write_png_rgb(path, width, height, bytes(img))

    return {
        "image": str(path),
        "width": width,
        "height": height,
        "cols": cols,
        "rows": rows,
        "cell_w": RC9IMG_CELL_W,
        "cell_h": RC9IMG_CELL_H,
        "glyph_cells": len(glyphs),
        "png_size": path.stat().st_size,
    }


def rc9_extract_all_glyphs_from_png(path: Path) -> Tuple[List[int], Dict[str, Any]]:
    width, height, rgb = rc9_read_png_rgb(path)
    if width % RC9IMG_CELL_W != 0 or height % RC9IMG_CELL_H != 0:
        raise ValueError("invalid RC9IMG dimensions")
    cols = width // RC9IMG_CELL_W
    rows = height // RC9IMG_CELL_H

    glyphs = []
    for i in range(cols * rows):
        glyphs.append(rc9_decode_glyph_cell(rgb, width, i % cols, i // cols))

    return glyphs, {
        "width": width,
        "height": height,
        "cols": cols,
        "rows": rows,
        "cell_w": RC9IMG_CELL_W,
        "cell_h": RC9IMG_CELL_H,
        "total_cells": cols * rows,
    }


def rc9img_encode_file_to_image(input_path: Path, output_png: Path, key_length: int = 25, cols: int = 192) -> Dict[str, Any]:
    source = input_path.read_bytes()
    rgs_blob, rgs_header = rgs_compress(source)

    source_digest = hashlib.sha256(source).digest()
    rgs_digest = hashlib.sha256(rgs_blob).digest()
    rgs_hash = rgs_digest.hex()

    glyphs, bit_padding = rc9_bytes_to_glyphs(rgs_blob)
    cubes, cube_pad_glyphs = rc9_glyphs_to_cubes(glyphs)

    flat_scrambled: List[int] = []
    for cube_index, cube in enumerate(cubes):
        key = rc9_generate_key(rgs_hash, cube_index=cube_index, length=key_length)
        scrambled = rc9_apply_sequence(cube, key)
        flat_scrambled.extend(rc9_state_to_flat(scrambled))

    payload_bytes = rc9_pack6(flat_scrambled)
    payload_digest = hashlib.sha256(payload_bytes).digest()

    header = RC9IMG_HDR.pack(
        RC9IMG_MAGIC,
        RC9IMG_VERSION_MAJOR,
        key_length,
        len(source),
        len(rgs_blob),
        bit_padding,
        cube_pad_glyphs,
        len(cubes),
        len(glyphs),
        len(flat_scrambled),
        source_digest,
        rgs_digest,
        payload_digest,
    )

    header_glyphs, header_pad = rc9_bytes_to_glyphs(header)
    if len(header_glyphs) != RC9IMG_HDR_GLYPHS or header_pad != RC9IMG_HDR_PAD:
        raise RuntimeError("header glyph calculation mismatch")

    image_glyphs = header_glyphs + flat_scrambled
    render_meta = rc9_render_glyph_png(output_png, image_glyphs, cols=cols)

    return {
        "ok": True,
        "system": RC9IMG_SYSTEM,
        "input": str(input_path),
        "output_png": str(output_png),
        "source_size": len(source),
        "rgs_mvgc_size": len(rgs_blob),
        "image_png_size": output_png.stat().st_size,
        "source_sha256": source_digest.hex(),
        "rgs_sha256": rgs_digest.hex(),
        "rgs_method": rgs_header.get("method"),
        "cube_count": len(cubes),
        "key_length": key_length,
        "glyph_count": len(glyphs),
        "payload_glyph_count": len(flat_scrambled),
        "header_glyph_count": len(header_glyphs),
        "total_image_glyphs": len(image_glyphs),
        "compression": {
            "rgs_ratio": round(len(rgs_blob) / max(1, len(source)), 6),
            "rgs_reduction_percent": round((1 - len(rgs_blob) / max(1, len(source))) * 100, 2),
            "image_ratio": round(output_png.stat().st_size / max(1, len(source)), 6),
            "image_reduction_percent": round((1 - output_png.stat().st_size / max(1, len(source))) * 100, 2),
            "image_factor": round(len(source) / max(1, output_png.stat().st_size), 4),
        },
        "render": render_meta,
    }


def rc9img_parse_image_header(all_glyphs: List[int]) -> Dict[str, Any]:
    if len(all_glyphs) < RC9IMG_HDR_GLYPHS:
        raise ValueError("image too small for RC9IMG header")
    header_bytes = rc9_glyphs_to_bytes(all_glyphs[:RC9IMG_HDR_GLYPHS], RC9IMG_HDR_PAD)[:RC9IMG_HDR.size]
    (
        magic,
        version_major,
        key_length,
        source_len,
        rgs_len,
        bit_padding,
        cube_pad_glyphs,
        cube_count,
        glyph_count,
        payload_glyph_count,
        source_digest,
        rgs_digest,
        payload_digest,
    ) = RC9IMG_HDR.unpack(header_bytes)

    if magic != RC9IMG_MAGIC:
        raise ValueError("bad RC9IMG magic")
    if version_major != RC9IMG_VERSION_MAJOR:
        raise ValueError(f"unsupported RC9IMG version: {version_major}")

    return {
        "system": RC9IMG_SYSTEM,
        "version_major": version_major,
        "key_length": key_length,
        "source_len": source_len,
        "rgs_len": rgs_len,
        "bit_padding": bit_padding,
        "cube_pad_glyphs": cube_pad_glyphs,
        "cube_count": cube_count,
        "glyph_count": glyph_count,
        "payload_glyph_count": payload_glyph_count,
        "source_sha256": source_digest.hex(),
        "rgs_sha256": rgs_digest.hex(),
        "payload_sha256": payload_digest.hex(),
        "header_glyph_count": RC9IMG_HDR_GLYPHS,
    }


def rc9img_decode_image_to_file(input_png: Path, output_path: Path) -> Dict[str, Any]:
    all_glyphs, img_meta = rc9_extract_all_glyphs_from_png(input_png)
    h = rc9img_parse_image_header(all_glyphs)

    start = int(h["header_glyph_count"])
    end = start + int(h["payload_glyph_count"])
    if len(all_glyphs) < end:
        raise ValueError("image truncated: not enough glyph cells")

    payload_glyphs = all_glyphs[start:end]
    payload_bytes = rc9_pack6(payload_glyphs)
    if rc9_sha256(payload_bytes) != h["payload_sha256"]:
        raise ValueError("RC9IMG payload SHA-256 mismatch")

    cube_count = int(h["cube_count"])
    if len(payload_glyphs) != cube_count * RC9IMG_CUBE_SIZE:
        raise ValueError("payload glyph count does not match cube count")

    restored_cubes = []
    rgs_hash = h["rgs_sha256"]
    key_length = int(h["key_length"])

    for cube_index in range(cube_count):
        part = payload_glyphs[cube_index * RC9IMG_CUBE_SIZE:(cube_index + 1) * RC9IMG_CUBE_SIZE]
        scrambled = rc9_flat_to_state(part)
        key = rc9_generate_key(rgs_hash, cube_index=cube_index, length=key_length)
        restored = rc9_apply_sequence(scrambled, rc9_invert_sequence(key))
        restored_cubes.append(restored)

    restored_glyphs = rc9_cubes_to_glyphs(restored_cubes, int(h["cube_pad_glyphs"]))
    if len(restored_glyphs) != int(h["glyph_count"]):
        raise ValueError("restored glyph count mismatch")

    rgs_blob = rc9_glyphs_to_bytes(restored_glyphs, int(h["bit_padding"]))[:int(h["rgs_len"])]
    if rc9_sha256(rgs_blob) != h["rgs_sha256"]:
        raise ValueError("RGS-MVGC blob SHA-256 mismatch")

    source, rgs_header = rgs_decompress(rgs_blob)
    if len(source) != int(h["source_len"]):
        raise ValueError("source length mismatch")
    if rc9_sha256(source) != h["source_sha256"]:
        raise ValueError("source SHA-256 mismatch")

    output_path.write_bytes(source)

    return {
        "ok": True,
        "system": RC9IMG_SYSTEM,
        "input_png": str(input_png),
        "output": str(output_path),
        "source_size": len(source),
        "source_sha256": rc9_sha256(source),
        "rgs_size": len(rgs_blob),
        "rgs_sha256": rc9_sha256(rgs_blob),
        "rgs_method": rgs_header.get("method"),
        "cube_count": cube_count,
        "key_length": key_length,
        "image": img_meta,
        "header": h,
    }


def rc9img_verify_roundtrip(png_path: Path, expected_source: Path | None = None) -> Dict[str, Any]:
    tmp = png_path.with_suffix(".decoded.tmp")
    try:
        result = rc9img_decode_image_to_file(png_path, tmp)
        out = {
            "ok": True,
            "png": str(png_path),
            "png_size": png_path.stat().st_size,
            "decoded_size": result["source_size"],
            "decoded_sha256": result["source_sha256"],
            "cube_count": result["cube_count"],
            "key_length": result["key_length"],
        }
        if expected_source is not None and expected_source.exists():
            expected_hash = rc9_sha256(expected_source.read_bytes())
            out["expected_sha256"] = expected_hash
            out["matches_expected"] = expected_hash == result["source_sha256"]
            if not out["matches_expected"]:
                raise ValueError("decoded PNG does not match expected source")
        return out
    finally:
        try:
            tmp.unlink()
        except FileNotFoundError:
            pass


# ============================================================
# Build the image-only transport container for the score zip.
# ============================================================

SUBMISSION_ZIP = Path("/kaggle/working/submission.zip")
RC9IMG_PNG = Path("/kaggle/working/submission.rc9img.png")
RC9IMG_DECODED_ZIP = Path("/kaggle/working/submission.rc9img.decoded.zip")
RC9IMG_MANIFEST = Path("/kaggle/working/RC9IMG_TRANSPORT_MANIFEST.json")

if not SUBMISSION_ZIP.exists():
    raise FileNotFoundError("submission.zip must exist before RC9IMG container generation")

encode_report = rc9img_encode_file_to_image(
    SUBMISSION_ZIP,
    RC9IMG_PNG,
    key_length=25,
    cols=192,
)

verify_report = rc9img_verify_roundtrip(
    RC9IMG_PNG,
    expected_source=SUBMISSION_ZIP,
)

decode_report = rc9img_decode_image_to_file(
    RC9IMG_PNG,
    RC9IMG_DECODED_ZIP,
)

if SUBMISSION_ZIP.read_bytes() != RC9IMG_DECODED_ZIP.read_bytes():
    raise RuntimeError("RC9IMG decoded zip differs from original submission.zip")

manifest = {
    "system": RC9IMG_SYSTEM,
    "status": "VERIFIED",
    "artifact_rule": "The PNG is the storage/transport container. No JSON or binary sidecar is needed to reconstruct submission.zip.",
    "competition_submission_file": str(SUBMISSION_ZIP),
    "rc9_image_container": str(RC9IMG_PNG),
    "decoded_check_file": str(RC9IMG_DECODED_ZIP),
    "encode_report": encode_report,
    "verify_report": verify_report,
    "decode_report": decode_report,
    "sha256": {
        "submission_zip": rc9_sha256(SUBMISSION_ZIP.read_bytes()),
        "rc9img_png": rc9_sha256(RC9IMG_PNG.read_bytes()),
        "decoded_zip": rc9_sha256(RC9IMG_DECODED_ZIP.read_bytes()),
    },
    "sizes": {
        "submission_zip": SUBMISSION_ZIP.stat().st_size,
        "rc9img_png": RC9IMG_PNG.stat().st_size,
        "decoded_zip": RC9IMG_DECODED_ZIP.stat().st_size,
    },
    "roundtrip": {
        "submission_zip_equals_decoded_zip": True,
    },
}

RC9IMG_MANIFEST.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("[RC9IMG] image-only container built and verified.")
print(json.dumps({
    "submission_zip": str(SUBMISSION_ZIP),
    "rc9img_png": str(RC9IMG_PNG),
    "manifest": str(RC9IMG_MANIFEST),
    "compression": encode_report["compression"],
    "roundtrip": manifest["roundtrip"],
}, indent=2, sort_keys=True))


## 7. Final listing

After this cell, open the **Output** tab. Submit `submission.zip` to Kaggle. Store or transport `submission.rc9img.png` as the RC9 image container.


In [7]:
!ls -lah /kaggle/working/submission.zip /kaggle/working/submission.rc9img.png /kaggle/working/submission.rc9img.decoded.zip /kaggle/working/RC9IMG_TRANSPORT_MANIFEST.json /kaggle/working/nemotron-adapter-ready-to-submit


-rw-r--r-- 1 root root 3.1G May  3 20:31 /kaggle/working/submission.zip

/kaggle/working/nemotron-adapter-ready-to-submit:
total 3.4G
drwxr-xr-x 2 root root 4.0K May  3 20:28 .
drwxr-xr-x 3 root root 4.0K May  3 20:28 ..
-rw-r--r-- 1 root root  618 May  3 20:28 adapter_config.json
-rw-r--r-- 1 root root 3.4G May  3 20:28 adapter_model.safetensors
-rw-r--r-- 1 root root    3 May  3 20:28 checkpoint_complete
-rw-r--r-- 1 root root 8.0K May  3 20:28 GLYPHMATICS_TRANSPORT_LEDGER.md
-rw-r--r-- 1 root root 8.1K May  3 20:28 README.md
